In [1]:
import os 
import numpy as np 

In [2]:

train_data = np.memmap("data/train.bin", dtype=np.uint16, mode="r")
val_data   = np.memmap("data/val.bin",   dtype=np.uint16, mode="r")

In [ ]:
import torch
block_size = 256
batch_size = 32
vocab_size = 4096
max_iters = 5000
eval_interval = 300
learning_rate = 3e-4
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 200
n_embd = 384
dropout = 0.2 
n_head = 4
n_layers = 6
def get_batch(split):
    data = train_data if split == "train" else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([torch.from_numpy(data[i : i + block_size].astype(np.int64))         for i in ix.tolist()])
    y = torch.stack([torch.from_numpy(data[i + 1 : i + 1 + block_size].astype(np.int64)) for i in ix.tolist()])
    return x.to(device), y.to(device)

In [ ]:
import torch.nn.functional as F
from torch import nn 

class GPTLanguageModel(nn.Module):
  def __init__(self):
    super().__init__()
    self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
    self.positional_embedding_table = nn.Embedding(block_size, n_embd)
    self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)])
    self.lm_head = nn.Linear(n_embd, vocab_size)

  def forward(self, idx, targets=None):
          B, T = idx.shape  # (B, T)
          tok_emb = self.token_embedding_table(idx) # (B,T, N_EMBD)
          pos_emb = self.positional_embedding_table(torch.arange(T, device = device)) # T,N_EMBD
          x = tok_emb + pos_emb
          x = self.blocks(x)
          logits = self.lm_head(x) # (B,T,vocab_size)

          loss = None
          if targets is not None:
              B, T, V = logits.shape
              loss = F.cross_entropy(logits.view(B*T, V), targets.view(B*T))
          return logits, loss
  @torch.no_grad()
  def generate(self, idx, max_new_tokens):
          # idx: (B, T) tensor of token ids to start from
          for _ in range(max_new_tokens):
              idx_cond = idx[:, -block_size:]
              logits, _ = self(idx_cond)
              logits = logits[:, -1, :]
              probs = F.softmax(logits, dim=-1)
              next_id = torch.multinomial(probs, 1)
              idx = torch.cat([idx, next_id], dim=1)
          return idx

In [5]:
class Block(nn.Module):
    """ Communication block followed by computation """
    def __init__(self,n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedForward(n_embd)
        self.ln1  = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self,x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x 

In [ ]:
class Head(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias = False)
        self.query = nn.Linear(n_embd, head_size, bias = False)
        self.value= nn.Linear(n_embd, head_size, bias = False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size,block_size)))
        self.dropout = nn.Dropout(dropout)
    def forward(self,x):
        B,T,C = x.shape
        k = self.key(x)
        q = self.query(x)
        v = self.value(x)

        wei = q @ k.transpose(-2,-1) * k.shape[-1] ** -0.5
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim = -1)
        wei = self.dropout(wei)
        out = wei @ v
        return out

In [ ]:
class MultiHeadAttention(nn.Module):
    """Multiple Attention Heads in parallel"""
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList(Head(head_size) for _ in range(num_heads))
        self.proj = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self,x):
        out = torch.cat([h(x) for h in self.heads], dim = -1)
        out = self.proj(out)
        return out

In [ ]:
class FeedForward(nn.Module):
    """ Feed Forward MLP"""
    def __init__(self,n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd,4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd,n_embd),
            nn.Dropout(dropout)
        )

    def forward(self,x):
        return self.net(x)

In [9]:
model = GPTLanguageModel().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

In [10]:
@torch.no_grad()
def estimate_loss():
        out = {}
        model.eval()
        for split in ["train", "val"]:
            losses = torch.zeros(eval_iters)
            for k in range(eval_iters):
                x, y = get_batch(split)
                _, loss = model(x, y)
                losses[k] = loss.item()
            out[split] = losses.mean().item()
        model.train()
        return out

In [11]:
for step in range(max_iters):

    if step % eval_interval == 0 or step == max_iters - 1:
        losses = estimate_loss()
        print(f"step {step}: train {losses['train']:.4f}, val {losses['val']:.4f}")

    x, y = get_batch("train")                # 1. get data
    logits, loss = model(x, y)               # 2. forward pass
    optimizer.zero_grad(set_to_none=True)    # 3. clear old gradients
    loss.backward()                          # 4. compute new gradients
    optimizer.step()                         # 5. update the table

step 0: train 8.4770, val 8.4740
step 300: train 4.6062, val 4.5870
step 600: train 4.2670, val 4.2759
step 900: train 4.1505, val 4.1425
step 1200: train 4.0354, val 4.0499
step 1500: train 3.9805, val 3.9944
step 1800: train 3.9306, val 3.8968
step 2100: train 3.8568, val 3.8613
step 2400: train 3.8278, val 3.8244
step 2700: train 3.7955, val 3.7640
step 3000: train 3.7723, val 3.8004
step 3300: train 3.7541, val 3.7384
step 3600: train 3.7270, val 3.7419
step 3900: train 3.7339, val 3.7336
step 4200: train 3.7137, val 3.7507
step 4500: train 3.6940, val 3.6665
step 4800: train 3.6547, val 3.7059
step 4999: train 3.6689, val 3.6680


In [12]:
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained("tokenizer")
EOT_ID = tok.convert_tokens_to_ids("<|endoftext|>")   # 0

start = torch.tensor([[EOT_ID]], dtype=torch.long, device=device)   # shape (1, 1)
out = model.generate(start, max_new_tokens=100)                     # shape (1, 101)
print(tok.decode(out[0].tolist()))

<|endoftext|>Once upon a time, there was a sheep. shut more dance in meat and How to keep the successful cookies. They are obedient girl.<|endoftext|>Once upon a time, there was a doctor named Mom. Bella and
veled the job, and she said, "We are stuck. Buddy, as you want to play a hole in water for it to wash the knife.<|endoftext|> remember to do the doll is mine."

"How and one!" Tom said smiling andOloe


In [13]:
os.makedirs("model", exist_ok = True)
torch.save(model.state_dict(), 'model/attention_with_residual_norm.pth')
torch.save(optimizer.state_dict(), 'model/optimizer_attention_with_residual_norm.pth')